# 07 — Insights do tráfego de acesso ao Porto de Santos

**Fontes:** volume pedagiado horário e volume de tráfego diário. **Não somar as duas bases**, pois podem descrever as mesmas passagens. O recorte de acesso exige a configuração validada das praças/lotes. Resultados não demonstram causalidade meteorológica.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd().resolve()
while ROOT.name != 'porto-santos':
    if ROOT==ROOT.parent: raise FileNotFoundError('Pasta porto-santos não encontrada')
    ROOT=ROOT.parent
DATA=ROOT/'data/processed/trafego'
REPORT=ROOT/'reports/trafego'
DESTINO=DATA
DESTINO.mkdir(parents=True,exist_ok=True)
print('Projeto:', ROOT)

## 1. Conferir os arquivos e o filtro regional

Se os arquivos tratados ainda não existirem, execute o ETL e revise `reports/trafego/inventario_pracas_lotes.csv`, preenchendo `data/config/trafego_pracas_acesso.csv` com valores **verificados**.

In [ ]:
import json
resumo_path=REPORT/'resumo_qualidade.json'
if resumo_path.exists():
    print(json.loads(resumo_path.read_text(encoding='utf-8')))
for nome in ['volume_pedagiado_acesso_horario.csv','volume_trafego_acesso_diario.csv']:
    p=DATA/nome
    print(nome, 'OK' if p.exists() else 'NÃO GERADO')

## 2. Selecionar uma única fonte

Comece pelo **diário** para análises anuais, mensais e eventual cruzamento por data. A fonte horária é analisada separadamente para padrões por hora.

In [ ]:
ARQUIVO=DATA/'volume_trafego_acesso_diario.csv'
if not ARQUIVO.exists(): raise FileNotFoundError('Execute clean_trafego.py com as praças/lotes validados. Veja o inventário em reports/trafego.')
df=pd.read_csv(ARQUIVO,parse_dates=['DATA'],low_memory=False)
print('Linhas:',len(df));display(df.head());display(df.isna().mean().sort_values(ascending=False).mul(100).round(2).to_frame('% ausente'))

## 3. Resumo diário por praça e sentido

Somente os volumes *informados* são somados. Quando não existe medição válida, os agregados permanecem ausentes.

In [ ]:
base=df.dropna(subset=['DATA']).copy()
colunas=['TOTAL_VEICULOS_INFORMADOS','VEICULOS_PESADOS_INFORMADOS','VEICULOS_LEVES_INFORMADOS','MOTOS_INFORMADAS']
diario=base.groupby(['DATA','PRACA','SENTIDO'],dropna=False)[colunas].sum(min_count=1).reset_index()
diario.to_csv(DESTINO/'trafego_acesso_resumo_diario.csv',index=False,encoding='utf-8-sig')
display(diario.head())

## 4. Evolução mensal por praça

A comparação anual deve considerar a abrangência das praças e a completude de cada ano; os arquivos podem representar malhas diferentes.

In [ ]:
diario['ANO']=diario.DATA.dt.year;diario['MES']=diario.DATA.dt.month
mensal=diario.groupby(['ANO','MES','PRACA'],dropna=False)[colunas].sum(min_count=1).reset_index()
mensal.to_csv(DESTINO/'trafego_acesso_resumo_mensal.csv',index=False,encoding='utf-8-sig')
display(mensal.head(15))

In [ ]:
serie=mensal.groupby(['ANO','MES'])['TOTAL_VEICULOS_INFORMADOS'].sum(min_count=1).reset_index()
serie['DATA']=pd.to_datetime(dict(year=serie.ANO,month=serie.MES,day=1))
plt.figure(figsize=(12,5));plt.plot(serie.DATA,serie.TOTAL_VEICULOS_INFORMADOS,marker='o');plt.title('Volume pedagiado diário agregado por mês — praças selecionadas');plt.ylabel('Veículos informados');plt.xlabel('Mês');plt.xticks(rotation=45);plt.grid(alpha=.3);plt.tight_layout();plt.show()

## 5. Participação dos veículos pesados

A participação é calculada sobre categorias de veículos informadas. Verifique a equivalência de categorias por período.

In [ ]:
mensal['PARTICIPACAO_PESADOS_%']=100*mensal.VEICULOS_PESADOS_INFORMADOS/mensal.TOTAL_VEICULOS_INFORMADOS.replace(0,pd.NA)
display(mensal[['ANO','MES','PRACA','PARTICIPACAO_PESADOS_%']].sort_values('PARTICIPACAO_PESADOS_%',ascending=False).head(20))
mensal.to_csv(DESTINO/'trafego_acesso_resumo_mensal.csv',index=False,encoding='utf-8-sig')

## 6. Variação entre dias e ranking de maiores volumes

In [ ]:
rank=diario.sort_values('TOTAL_VEICULOS_INFORMADOS',ascending=False).head(20)
display(rank[['DATA','PRACA','SENTIDO','TOTAL_VEICULOS_INFORMADOS']])
rank.to_csv(DESTINO/'trafego_acesso_dias_maior_volume.csv',index=False,encoding='utf-8-sig')

## 7. Inspecionar cobertura temporal

Dias ausentes podem refletir não operação ou falhas; o painel deve evidenciar as lacunas, não substituí-las por zero.

In [ ]:
cobertura=base.groupby(['PRACA',base.DATA.dt.year.rename('ANO')]).agg(dias_com_registros=('DATA','nunique'),linhas=('DATA','size')).reset_index()
display(cobertura);cobertura.to_csv(DESTINO/'trafego_acesso_cobertura_anual.csv',index=False,encoding='utf-8-sig')

## 8. Fonte horária (análise separada)

Jamais junte seus volumes aos totais da fonte diária.

In [ ]:
p=DATA/'volume_pedagiado_acesso_horario.csv'
if p.exists():
    horas=pd.read_csv(p,usecols=['DATA','HORA','PRACA','TOTAL_VEICULOS_INFORMADOS'],low_memory=False)
    horas['HORA']=horas.HORA.astype('string').str.extract(r'(\d{1,2})',expand=False)
    horas['HORA']=pd.to_numeric(horas.HORA,errors='coerce')
    perfil=horas.groupby(['PRACA','HORA'])['TOTAL_VEICULOS_INFORMADOS'].sum(min_count=1).reset_index()
    display(perfil.head(24));perfil.to_csv(DESTINO/'trafego_acesso_perfil_horario.csv',index=False,encoding='utf-8-sig')
else: print('Sem base horária filtrada.')

## Entregáveis para o Power BI

- `volume_trafego_acesso_diario.csv`: microdados tratados da fonte diária.
- `volume_pedagiado_acesso_horario.csv`: microdados tratados horários (se disponível).
- `trafego_acesso_resumo_diario.csv`: agregação por data, praça e sentido.
- `trafego_acesso_resumo_mensal.csv`: acompanhamento mensal.
- `trafego_acesso_cobertura_anual.csv`: qualidade temporal.

**Interpretação:** diferenças de volume podem resultar de alterações na cobertura de praças/lotes, calendário, demanda e outros fatores; não atribuir causalidade ao clima sem cruzamento válido.